In [2]:
import polars as pl
from deltalake import DeltaTable

In [3]:
from config.paths import LAKE_ROOT

lake_path = str(LAKE_ROOT / "silver_layer")


In [4]:
face = pl.scan_delta(f"{lake_path}/face_processos_clean_delta")

### Metodologia (top 500 no período)

**Fonte de dados.** Registos provenientes da camada silver, tabela Delta de processos face limpos (`face_processos_clean_delta`).

**Critério de tramitação (substitui a janela entre datas).** Incluem-se observações com `tempo_tramitacao_meses` não ausente, não negativos (coerente com duração observável) e inferiores ou iguais a 60 meses (cinco anos de tramitação aproximados nessa coluna do silver).

**Recorte temporal.** Cria-se a variável de ano a partir de `distribuicao_data` (ano civil da data de distribuição). Guardam-se apenas processos cujo ano está entre 2020 e 2025, inclusive.

**Regra de seleção global.** No conjunto inteiro desse intervalo (sem estratificar por ano), ordena-se por `valor_corrigido_atual` em **ordem decrescente** — da causa mais cara para a menos cara. Valores nulos ficam no fim. Recolhem-se, no máximo, **500** linhas totais no período.

**Desempates e ordem final.** Com empates no valor corrigido, a ordem segue a convenção do motor. O resultado final mantém ordenação principal por `valor_corrigido_atual` (decrescente) e, como critério secundário para leitura, `ano` (crescente).

**Nota operativa.** A etapa de filtro na tabela lê-se em *lazy* com *streaming* quando possível; a extração final das 500 linhas ocorre após o materializador necessário.


In [5]:
# Top 500 processos no período 2020–2025 (sem separar por ano), com tramitação <= 60 meses
# Ano = ano civil de `distribuicao_data` (ajuste a coluna abaixo se quiser p.ex. data da sentença)

VAL = "valor_corrigido_atual"
ANOS = (2020, 2025)  # inclusive: 2020 … 2025
MAX_TRAMIT_MESES = 60
N_TOTAL = 500

data_para_ano = pl.col("distribuicao_data").cast(pl.Datetime("us"))

base = (
    face
    .with_columns(ano=data_para_ano.dt.year())
    .filter(pl.col("ano").is_between(ANOS[0], ANOS[1]))
    .filter(
        pl.col("tempo_tramitacao_meses").is_not_null()
        & (pl.col("tempo_tramitacao_meses") >= 0)
        & (pl.col("tempo_tramitacao_meses") <= MAX_TRAMIT_MESES)
    )
    .collect(engine="streaming")
)

amostra_top500_janela = (
    base
    .sort([VAL, "ano"], descending=[True, False], nulls_last=True)
    .head(N_TOTAL)
)

In [6]:
# Exportar para CSV (UTF-8; altere o caminho se precisar)
# Excel em PT: troque a última linha p.ex. separador: amostra_top500_janela.write_csv(CSV_OUT, separator=";")
CSV_OUT = "amostra_2020_2025_top500_face.csv"
amostra_top500_janela.write_csv(CSV_OUT)
# OPCIONAL — Parquet:
# amostra_top500_janela.write_parquet("amostra_2020_2025_top500_face.parquet")

In [7]:
amostra_top500_janela.height, amostra_top500_janela.group_by("ano").len().sort("ano")

(500,
 shape: (6, 2)
 ┌──────┬─────┐
 │ ano  ┆ len │
 │ ---  ┆ --- │
 │ i32  ┆ u32 │
 ╞══════╪═════╡
 │ 2020 ┆ 212 │
 │ 2021 ┆ 100 │
 │ 2022 ┆ 62  │
 │ 2023 ┆ 45  │
 │ 2024 ┆ 15  │
 │ 2025 ┆ 66  │
 └──────┴─────┘)

In [8]:
amostra_top500_janela.head(10)

numero,classe,assunto,foro,vara,distribuicao,controle,area,valor,outros_assuntos,autores,advogados_autores,reus,advogados_reus,movimentações,tipo_sentença,data_sentença,cd_processo,url_scraped,ingested_at,num_processo_limpo,distribuicao_data,valor_limpo,data_sentenca_clean,tempo_tramitacao_meses,valor_corrigido_atual,valor_sm,updated_at,fator_correcao_ipca,ano
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,"datetime[μs, UTC]",str,datetime[μs],f64,datetime[μs],f64,f64,f64,"datetime[μs, UTC]",f64,i32
"""0101256-29.2005.8.26.0222""","""Execução Fiscal""","""PIS""","""Foro 12 - Núcleo 4.0""","""Unidade 12 - Núcleo 4.0 Execuç…","""12/08/2025 às 01:19 - Direcion…","""2025/038842""","""Cível""","""R$ 684.443.650,00""","""Cível""","""União Federal - PRFN""","""NÃO HÁ REGISTRO""","""MARCIO DINIZ GOTLIB, SOCRATES …","""Jorge Antonio Ioriatti Chami, …","""[{""data"": ""13/02/2026"", ""tipo""…","""[""Extinto o Processo pelo Canc…","""[""18/12/2025"", ""29/11/2025"", ""…","""66Z100PI90000""","""https://esaj.tjsp.jus.br/cpopg…",2026-04-06 18:25:11.533827 UTC,"""01012562920058260222""",2025-08-12 00:00:00,6.8444365e8,2025-12-18 00:00:00,4.2,6.9824e8,430213.1,2026-04-08 13:36:44.523003 UTC,1.020151,2025
"""0017575-65.2010.8.26.0068""","""Execução Fiscal""","""ICMS/ Imposto sobre Circulação…","""Foro 8 - Núcleo 4.0""","""Unidade 8 - Núcleo 4.0 Execuçõ…","""12/08/2025 às 00:33 - Direcion…","""2025/008744""","""Cível""","""R$ 620.590.456,00""","""Cível""","""Fazenda Pública do Estado de S…","""NÃO HÁ REGISTRO""","""Professional Network do Brasil…","""Ivan Henrique Moraes Lima, Leo…","""[{""data"": ""31/03/2026"", ""tipo""…","""[""Proferidas Outras Decisões n…","""[""06/03/2026"", ""18/06/2025"", ""…","""1WZ1NN43V0000""","""https://esaj.tjsp.jus.br/cpopg…",2026-04-07 01:37:34.426554 UTC,"""00175756520108260068""",2025-08-12 00:00:00,6.20590456e8,2026-03-06 00:00:00,6.77,6.3310e8,390077.61,2026-04-08 13:36:01.289623 UTC,1.020151,2025
"""0070995-31.2012.8.26.0224""","""Execução Fiscal""","""ICMS/ Imposto sobre Circulação…","""Foro de Guarulhos""","""SETOR DE EXECUÇÕES FISCAIS DA …","""28/11/2020 às 12:39 - Direcion…","""2012/019170""","""Cível""","""R$ 146.654.509,99""","""Cível""","""Fazenda do Estado de São Paulo""","""Elisabete Nunes Guardado""","""Vaska Industria e Comercio de …","""NÃO HÁ REGISTRO""","""[{""data"": ""27/02/2025"", ""tipo""…","""[""Proferidas Outras Decisões n…","""[""26/07/2023"", ""28/06/2023""]""","""68Z0C1IS30000""","""https://esaj.tjsp.jus.br/cpopg…",2026-04-06 18:49:56.268522 UTC,"""00709953120128260224""",2020-11-28 00:00:00,1.4665e8,2023-07-26 00:00:00,31.87,2.0171e8,124283.21,2026-04-08 13:35:09.832508 UTC,1.375421,2020
"""1623943-69.2021.8.26.0090""","""Execução Fiscal""","""ISS/ Imposto sobre Serviços""","""Foro das Execuções Fiscais Mun…","""Vara das Execuções Fiscais Mun…","""08/07/2021 às 00:42 - Livre""","""2021/125596""","""Cível""","""R$ 118.796.256,00""","""Cível""","""PREFEITURA MUNICIPAL DE SÃO PA…","""NÃO HÁ REGISTRO""","""Banco Bv S.a.""","""Maria Rita Ferragut, Juliana d…","""[{""data"": ""24/02/2025"", ""tipo""…","""[""Extinta a Execução/Cumprimen…","""[""02/12/2024"", ""24/08/2023""]""","""2I000QUAB0000""","""https://esaj.tjsp.jus.br/cpopg…",2026-04-07 00:36:41.911857 UTC,"""16239436920218260090""",2021-07-08 00:00:00,1.18796256e8,2024-12-02 00:00:00,40.83,1.5400e8,94884.36,2026-04-08 13:35:05.614863 UTC,1.296314,2021
"""1510783-91.2021.8.26.0114""","""Execução Fiscal""","""ISS/ Imposto sobre Serviços""","""Foro de Campinas""","""SEF - Setor de Execuções Fisca…","""28/05/2021 às 12:25 - Livre""","""2021/007429""","""Cível""","""R$ 93.485.260,20""","""Cível""","""MUNICÍPIO DE CAMPINAS""","""NÃO HÁ REGISTRO""","""Unimed Campinas Cooperativa de…","""Jose Luis Finocchio Junior, Ju…","""[{""data"": ""23/07/2025"", ""tipo""…","""[""Extinta a Execução/Cumprimen…","""[""05/06/2025""]""","""36000T5QB0000""","""https://esaj.tjsp.jus.br/cpopg…",2026-04-07 11:34:47.959417 UTC,"""1510783

In [9]:
amostra_top500_janela.

SyntaxError: invalid syntax (3128062761.py, line 1)